In [ ]:
pip install requests

In [ ]:
!pip install kafka-python-ng

In [ ]:
pip install python-dotenv

In [ ]:
import os
import requests
import json
import time
from kafka import KafkaProducer
from dotenv import load_dotenv

# Load environmental variables from the .env file
load_dotenv("../.env")

# Retrieve the OpenWeather API key from environment variables
API_KEY = os.getenv("OPENWEATHER_API_KEY")

try:
    # Initialize Kafka Producer with integrated retry logic to handle temporary connection drops
    producer = KafkaProducer(
        bootstrap_servers=['127.0.0.1:9092'], # تم تعديل دي عشان نحل مشكلة الويندوز
        retries=5,                          # Automatically retry up to 5 times if broker is unavailable
        retry_backoff_ms=2000,              # Wait for 2 seconds (2000ms) between each retry attempt
        value_serializer=lambda v: json.dumps(v).encode('utf-8')  # Serialize python dictionary to JSON binary
    )
    print("Kafka Producer initialized successfully!")
except Exception as e:
    print(f"Failed to connect to Kafka: {e}")
    producer = None

def stream_weather_by_coordinates(lat, lon):
    # Construct the OpenWeather API URL with coordinates and metric units
    URL = f"http://api.openweathermap.org/data/2.5/weather?lat={lat}&lon={lon}&appid={API_KEY}&units=metric"
    
    try:
        # Fetch live weather data from the API
        response = requests.get(URL)
        data = response.json()

        # Process the response if the API call was successful (HTTP 200)
        if response.status_code == 200:
            # Build the unified data payload matching our data model schema
            payload = {
                "city": data.get("name", "Unknown Location"), 
                "latitude": lat,
                "longitude": lon,
                "temp": data['main']['temp'],
                "humidity": data['main']['humidity'],
                "pressure": data['main']['pressure'],
                "sea_level": data['main'].get('sea_level', 0),
                "wind_speed": data['wind']['speed'],
                "clouds": data['clouds']['all'],
                "rain_1h": data.get('rain', {}).get('1h', 0.0),
                "weather_description": data['weather'][0]['description'],
                "timestamp": time.time()
            }

            print(f"Location [{lat}, {lon}] -> City: {payload['city']} | Temp: {payload['temp']}°C | Humidity: {payload['humidity']}%")
            
            # Publish the parsed payload to the Kafka topic if the producer is active
            if producer:
                producer.send('weather_data', value=payload)
                producer.flush()  # Ensure data is completely sent before proceeding
                print("Data pushed to Kafka successfully!")
        else:
            print(f"API Error for [{lat}, {lon}]: {data.get('message')}")

    except Exception as e:
        print(f"Unexpected error during streaming: {e}")

# Target coordinates for streaming testing (Minya, Egypt)
test_lat = 28.0871
test_lon = 30.7618

print("Starting live weather stream... Press Ctrl+C to stop.")

# Run continuously to simulate a real-time stream
try:
    while True:
        # Trigger the live weather streaming execution
        stream_weather_by_coordinates(test_lat, test_lon)
        
        # Wait for 60 seconds before fetching the next data point
        time.sleep(60)
except KeyboardInterrupt:
    print("\nStreaming stopped by user.")
finally:
    if producer:
        producer.close()
        print("Kafka Producer closed safely.")

Kafka Producer initialized successfully!
Starting live weather stream... Press Ctrl+C to stop.
Location [28.0871, 30.7618] -> City: Minya | Temp: 37.74°C | Humidity: 15%
Data pushed to Kafka successfully!
Location [28.0871, 30.7618] -> City: Minya | Temp: 37.74°C | Humidity: 15%
Data pushed to Kafka successfully!
Location [28.0871, 30.7618] -> City: Minya | Temp: 37.74°C | Humidity: 15%
Data pushed to Kafka successfully!
Location [28.0871, 30.7618] -> City: Minya | Temp: 37.74°C | Humidity: 15%
Data pushed to Kafka successfully!
Location [28.0871, 30.7618] -> City: Minya | Temp: 37.74°C | Humidity: 15%
Data pushed to Kafka successfully!
Location [28.0871, 30.7618] -> City: Minya | Temp: 37.74°C | Humidity: 15%
Data pushed to Kafka successfully!
Location [28.0871, 30.7618] -> City: Minya | Temp: 37.74°C | Humidity: 15%
Data pushed to Kafka successfully!
Location [28.0871, 30.7618] -> City: Minya | Temp: 37.74°C | Humidity: 15%
Data pushed to Kafka successfully!
Location [28.0871, 30.761